# Telemetry analysis

Post-hoc analysis of a binary telemetry capture. The live view
(`capture.py --live`) answers *"is it sane right now?"*; this notebook
answers *"why did it do that?"* — full rate, every field.

**Kernel:** select `moteus-venv` (Python 3, needs pandas + pyarrow).

```bash
# capture (Windows python, invoked from the WSL terminal)
python.exe tools/telemetry/capture.py --port COM9 --out logs/t.bin -s 60
```

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The tools live beside this notebook; make them importable regardless of
# where Jupyter was launched from.
HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import decode
import scale

# ---------------------------------------------------------------- palette
#
# Validated categorical palette, assigned in FIXED SLOT ORDER and never
# cycled.  Colour identifies a series; it never encodes magnitude, and all
# text stays in ink tokens so identity never rests on colour alone.
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"
C_YELLOW, C_MAGENTA = "#eda100", "#e87ba4"
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#d8d7d2"

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK_MUTED,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "medium",
    "axes.titlelocation": "left",
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.7,
    "lines.linewidth": 1.6,
    "text.color": INK,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "legend.frameon": False,
})

## Load

Point `LOG` at a capture. The decode summary is printed in full: frames
decoded, CRC failures, resync bytes and loss percentage. **Read it before
trusting any plot below** — a high loss rate changes what the data means,
and it is reported rather than hidden precisely so it cannot be missed.

In [ ]:
# Newest capture in logs/, or set the path explicitly.
candidates = sorted((HERE.parent.parent / "logs").glob("telemetry_*.bin"))
LOG = candidates[-1] if candidates else None

if LOG is None:
    raise SystemExit("No logs/telemetry_*.bin found -- run capture.py first.")

blob = LOG.read_bytes()
stats = decode.DecodeStats()
records = list(decode.decode_bytes(blob, stats))

print(f"=== {LOG.name} ===")
print(stats.report(len(blob)))

df = decode.to_dataframe(records)
duration = df["t_s"].iloc[-1]
print(f"  duration        {duration:.2f} s")
print(f"  mean rate       {(len(df) - 1) / duration:.1f} Hz")
df.head()

## 1 — Control tracking error

Commanded vs measured torque, and the difference. Both live in the same
frame, which is the only reason the error is computable at all.

Note there is **no dual y-axis** anywhere in this notebook: two measures on
one plot with two scales is the single most misleading thing a chart can
do. Different units get different panels.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

ax = axes[0]
ax.plot(df["t_s"], df["cmd_torque_nm"], color=C_BLUE, label="commanded")
ax.plot(df["t_s"], df["motor_torque_nm"], color=C_ORANGE, label="measured")
ax.set_ylabel("torque (Nm)")
ax.set_title("Torque tracking")
ax.legend(loc="upper right", ncol=2)

# Shade the spans where the clamp actually limited the command.  Persistent
# saturation means the gains are too hot or the cube is past recovery.
clamped = df["flag_torque_clamped"].to_numpy()
if clamped.any():
    ax.fill_between(df["t_s"], *ax.get_ylim(), where=clamped,
                    color=C_YELLOW, alpha=0.15, step="mid",
                    label="clamped")
    ax.legend(loc="upper right", ncol=3)

ax = axes[1]
ax.plot(df["t_s"], df["tracking_error_nm"], color=C_AQUA)
ax.axhline(0.0, color=INK_MUTED, linewidth=0.8, linestyle="--")
ax.set_ylabel("error (Nm)")
ax.set_title("Tracking error  (commanded - measured)")

ax = axes[2]
ax.plot(df["t_s"], np.degrees(df["theta"]), color=C_BLUE)
ax.axhline(0.0, color=INK_MUTED, linewidth=0.8, linestyle="--")
ax.set_ylabel("tilt (deg)")
ax.set_xlabel("time (s)")
ax.set_title("Tilt about the balance point")

fig.tight_layout()
plt.show()

rms = float(np.sqrt(np.mean(df["tracking_error_nm"] ** 2)))
print(f"RMS tracking error   {rms:.4f} Nm")
print(f"clamped cycles       {int(clamped.sum())} / {len(df)}")

## 2 — Control terms

Which gain is doing the work. These are recorded **pre-clamp**, so a term
that dwarfs the others here is the one to retune — visible even when the
output was saturated and the final torque tells you nothing.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(df["t_s"], df["term_theta"], color=C_BLUE, label=r"$k_\theta \cdot \theta$")
ax.plot(df["t_s"], df["term_theta_dot"], color=C_ORANGE,
        label=r"$k_{\dot\theta} \cdot \dot\theta$")
ax.plot(df["t_s"], df["term_omega"], color=C_AQUA, label=r"$k_\omega \cdot \omega$")
ax.axhline(0.0, color=INK_MUTED, linewidth=0.8, linestyle="--")
ax.set_xlabel("time (s)")
ax.set_ylabel("torque contribution (Nm)")
ax.set_title("Control terms, pre-clamp")
ax.legend(loc="upper right", ncol=3)
fig.tight_layout()
plt.show()

for name, col in [("theta", "term_theta"), ("theta_dot", "term_theta_dot"),
                  ("omega", "term_omega")]:
    print(f"  {name:10s} RMS {float(np.sqrt(np.mean(df[col] ** 2))):.4f} Nm")

## 3 — IMU dynamic response

Raw counts and calibrated SI overlaid. They should differ **only** by the
datasheet scale factor — `decode.to_dataframe()` already asserted that, and
prints a warning if the firmware's range register and `scale.py` disagree.

The gyro PSD shows where the noise actually lives, which is what sets a
sensible complementary-filter time constant.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6.5))

ax = axes[0][0]
for col, colour, label in [("accel_x", C_BLUE, "x"),
                           ("accel_y", C_ORANGE, "y"),
                           ("accel_z", C_AQUA, "z")]:
    ax.plot(df["t_s"], df[col], color=colour, label=label)
ax.set_ylabel("accel (m/s$^2$)")
ax.set_title("Accelerometer, calibrated")
ax.legend(loc="upper right", ncol=3)

ax = axes[0][1]
for col, colour, label in [("gyro_x", C_BLUE, "x"),
                           ("gyro_y", C_ORANGE, "y"),
                           ("gyro_z", C_AQUA, "z")]:
    ax.plot(df["t_s"], df[col], color=colour, label=label)
ax.set_ylabel("gyro (rad/s)")
ax.set_title("Gyroscope, calibrated")
ax.legend(loc="upper right", ncol=3)

# Magnitude: at rest this must sit at 9.81.  A steady offset is a
# calibration error; a wandering one is real motion.
ax = axes[1][0]
mag = np.sqrt(df["accel_x"] ** 2 + df["accel_y"] ** 2 + df["accel_z"] ** 2)
ax.plot(df["t_s"], mag, color=C_BLUE, label="|a|")
ax.axhline(scale.GRAVITY, color=INK_MUTED, linewidth=0.9, linestyle="--")
ax.annotate("9.81", xy=(df["t_s"].iloc[-1], scale.GRAVITY),
            xytext=(-28, 6), textcoords="offset points",
            color=INK_MUTED, fontsize=9)
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (m/s$^2$)")
ax.set_title("Acceleration magnitude")

# Power spectral density of the gyro.  Log-log, one series per axis.
ax = axes[1][1]
dt = float(np.median(np.diff(df["t_s"])))
fs = 1.0 / dt if dt > 0 else 400.0
for col, colour, label in [("gyro_x", C_BLUE, "x"),
                           ("gyro_y", C_ORANGE, "y"),
                           ("gyro_z", C_AQUA, "z")]:
    signal = df[col].to_numpy() - df[col].mean()
    freqs = np.fft.rfftfreq(len(signal), d=dt)
    psd = np.abs(np.fft.rfft(signal)) ** 2 / (fs * len(signal))
    ax.loglog(freqs[1:], psd[1:], color=colour, linewidth=1.0, label=label)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("PSD (rad$^2$/s$^2$/Hz)")
ax.set_title(f"Gyro noise spectrum  (fs = {fs:.0f} Hz)")
ax.legend(loc="upper right", ncol=3)

fig.tight_layout()
plt.show()

print(f"accel quantisation  {scale.accel_resolution_ms2():.5f} m/s^2")
print(f"gyro  quantisation  {scale.gyro_resolution_rad_s():.6f} rad/s")
print(f"|a| mean {float(mag.mean()):.4f}  sd {float(mag.std()):.4f} m/s^2")

## 4 — Gyro bias vs temperature

The BMI270 datasheet gives the gyro zero-rate offset a drift of
**±0.015 dps/K**. That is why die temperature rides in every frame: without
it, a slow bias drift and a genuine slow rotation are the same signal.

On a stationary capture the gyro mean *is* the bias, so a slope here is the
drift coefficient measured on your actual part. If the board moved during
the capture this plot is meaningless — check `|a|` above first.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

ax = axes[0]
ax.plot(df["t_s"], df["imu_temp_c"], color=C_ORANGE)
ax.set_xlabel("time (s)")
ax.set_ylabel("die temp (deg C)")
ax.set_title("IMU die temperature")

# Bias estimate over a sliding window, against temperature.
ax = axes[1]
window = max(int(len(df) / 50), 10)
temp = df["imu_temp_c"].rolling(window).mean()
for col, colour, label in [("gyro_x", C_BLUE, "x"),
                           ("gyro_y", C_ORANGE, "y"),
                           ("gyro_z", C_AQUA, "z")]:
    bias = df[col].rolling(window).mean()
    ax.plot(temp, np.degrees(bias), color=colour, linewidth=1.2, label=label)
ax.set_xlabel("die temp (deg C)")
ax.set_ylabel("gyro bias (dps)")
ax.set_title(f"Bias vs temperature  (rolling {window} samples)")
ax.legend(loc="upper right", ncol=3)

fig.tight_layout()
plt.show()

span = float(df["imu_temp_c"].max() - df["imu_temp_c"].min())
print(f"temperature span {span:.2f} K")
if span < 1.0:
    print("  Too small to fit a drift coefficient -- the datasheet figure is")
    print("  0.015 dps/K, so you need several K of span to see it. Let the")
    print("  board warm up from cold, or run a longer capture.")

## 5 — Driver power

Bus voltage times q-axis current. Note `q_current_a` is **0 until
`MoteusDriverWrapper::FromQuery` is extended** — it currently drops
`q_current`, `d_current`, `power` and board temperature from
`Query::Result`. Until then this panel is structurally correct and
numerically empty.

In [ ]:
if df["q_current_a"].abs().max() == 0:
    print("q_current is all zero -- FromQuery has not been extended yet, or")
    print("this capture came from telemetry_test (no CAN link). Skipping.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.4), sharex=True)

    ax = axes[0]
    ax.plot(df["t_s"], df["power_w"], color=C_BLUE)
    ax.axhline(0.0, color=INK_MUTED, linewidth=0.8, linestyle="--")
    ax.set_ylabel("power (W)")
    ax.set_title("Instantaneous electrical power  (bus V x q-axis A)")

    ax = axes[1]
    energy = np.cumsum(df["power_w"].to_numpy() * df["dt_s"].to_numpy())
    ax.plot(df["t_s"], energy, color=C_ORANGE)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("energy (J)")
    ax.set_title("Cumulative energy")

    fig.tight_layout()
    plt.show()

    print(f"mean power {float(df['power_w'].mean()):.2f} W   "
          f"peak {float(df['power_w'].abs().max()):.2f} W   "
          f"total {float(energy[-1]):.1f} J")

## 6 — Loop health

Did the loop hold its deadline? `dt_s` is the measured interval and
`slack_s` is the margin — negative slack is an overrun.

This is the panel that tells you whether the *timing* is trustworthy, and
therefore whether anything above it is.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

ax = axes[0]
dt_ms = df["dt_s"].to_numpy() * 1000.0
ax.hist(dt_ms, bins=60, color=C_BLUE, edgecolor="white", linewidth=0.5)
ax.axvline(float(np.median(dt_ms)), color=INK_MUTED, linewidth=1.0,
           linestyle="--")
ax.set_xlabel("cycle period (ms)")
ax.set_ylabel("count")
ax.set_title("Loop period distribution")

ax = axes[1]
slack_ms = df["slack_s"].to_numpy() * 1000.0
ax.plot(df["t_s"], slack_ms, color=C_AQUA, linewidth=1.0)
ax.axhline(0.0, color=INK_MUTED, linewidth=0.9, linestyle="--")
overruns = slack_ms < 0
if overruns.any():
    ax.scatter(df["t_s"].to_numpy()[overruns], slack_ms[overruns],
               s=14, color=C_ORANGE, zorder=3, label="overrun")
    ax.legend(loc="lower right")
ax.set_xlabel("time (s)")
ax.set_ylabel("slack (ms)")
ax.set_title("Deadline margin")

fig.tight_layout()
plt.show()

print(f"median period  {float(np.median(dt_ms)):.4f} ms")
print(f"p99 period     {float(np.percentile(dt_ms, 99)):.4f} ms")
print(f"worst period   {float(dt_ms.max()):.4f} ms")
print(f"overruns       {int(overruns.sum())} / {len(df)}"
      f"  ({100.0 * overruns.sum() / len(df):.2f}%)")
print(f"ring overflows {int(df['flag_overflow'].sum())}")

## 7 — Link quality

The measurement that decides whether WiFi telemetry is trustworthy for
tuning or only for monitoring. Run the same profile over USB and over UDP
and compare these numbers — USB is the control, so any excess loss is the
radio.

In [ ]:
expected = stats.frames_ok + stats.frames_lost
loss_pct = 100.0 * stats.frames_lost / expected if expected else 0.0

print(f"frames decoded   {stats.frames_ok}")
print(f"frames lost      {stats.frames_lost}  ({loss_pct:.3f}%)")
print(f"CRC failures     {stats.crc_errors}")
print(f"resync bytes     {stats.resync_bytes}")
print(f"ring overflows   {stats.overflow_frames}")
print()
if stats.crc_errors == 0 and stats.frames_lost == 0:
    print("Clean link.  Over USB this is the expected result -- any loss here")
    print("is a bug in the firmware drain, not the transport.")
elif stats.overflow_frames > 0:
    print("Loss originated at the PRODUCER: the ring overflowed because the")
    print("drain could not keep up. Raise the drain budget or the ring size --")
    print("this is not a link problem.")
else:
    print("Loss originated on the LINK: frames left the ring but never")
    print("arrived. Expected on WiFi; on USB it means something is wrong.")

# Where in time was the loss?  Bursty loss and uniform loss have different
# causes -- a burst is usually a WiFi retry storm, uniform loss is
# saturation.
gaps = np.diff(df["seq"].to_numpy().astype(np.int64))
if (gaps > 1).any():
    fig, ax = plt.subplots(figsize=(11, 2.6))
    ax.scatter(df["t_s"].to_numpy()[1:][gaps > 1], gaps[gaps > 1],
               s=16, color=C_ORANGE)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("frames lost")
    ax.set_title("Loss events over time")
    fig.tight_layout()
    plt.show()